# Convolutional Neural Networks, Part 3: Regularization and Going Deeper

CSCI 6379 · Topic 22.

Part 2 ended on an uncomfortable number: our CNN reached **98.8% on the training set but only 74.3% on the test set**. It had memorised the 50,000 training images almost perfectly and was still wrong about one test image in four. That gap is the subject of this notebook.

We work through the three standard tools for closing it, each mirrored on the real course experiment code, then an appendix on ResNet:

- **Overfitting** — what the train/test gap looks like, why validation *loss* is the early warning.
- **Dropout** — randomly switching units off during training.
- **BatchNorm** — normalising each layer's output over the mini-batch.
- **Early stopping** — free, and most people forget it.
- **Appendix: ResNet** — the *degradation problem* and the residual connection.

**Runtime.** This notebook trains several small CNNs on CIFAR-10. Use a GPU runtime (Colab: *Runtime → Change runtime type → GPU*). Every experiment exposes an `EPOCHS` knob; the course numbers were measured at 30 epochs, and the defaults here are smaller so the notebook finishes quickly. Raise them to reproduce the reported figures.

## Setup: CIFAR-10

The same normalisation and loaders used in the course experiment scripts. On Colab we let `torchvision` download CIFAR-10 the first time.

In [ ]:
import os, copy, time
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

dev = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0); np.random.seed(0)
print("device =", dev)

# Same channel statistics as the course scripts (experiments3.py / experiments4.py).
tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

root = "./data"
tr = datasets.CIFAR10(root, train=True,  download=True, transform=tf)
te = datasets.CIFAR10(root, train=False, download=True, transform=tf)

# num_workers=2 matches the scripts; on Colab it is fine, set to 0 if you hit issues.
train_loader = DataLoader(tr, batch_size=128, shuffle=True,  num_workers=2)
test_loader  = DataLoader(te, batch_size=512, shuffle=False, num_workers=2)
print(f"train images = {len(tr):,}   test images = {len(te):,}")

### The Part 2 CNN, and one training loop for everything

A single flexible CNN so every experiment differs by exactly one variable. It is the four-convolution net from Part 2: two `3x3` conv blocks at 32 channels, max-pool, two at 64 channels, max-pool, then a small classifier head.

- `use_bn=False, p_drop=0.0` is the **plain Part 2 CNN** (2,168,362 parameters, our overfitting baseline).
- `p_drop=0.5` adds **dropout** in the classifier head — no extra parameters.
- `use_bn=True` inserts **`nn.BatchNorm2d`** after every convolution (adds 384 parameters, the per-channel gamma/beta pairs).

The `blk` helper mirrors `CNN_BN` in `experiments3.py`. We set `bias=False` on a convolution only when BatchNorm follows it, because BatchNorm subtracts the mean immediately afterwards and cancels any bias.

In [ ]:
class CNN(nn.Module):
    """The Part 2 CNN. Toggle BatchNorm and dropout to isolate one variable at a time."""
    def __init__(self, use_bn=False, p_drop=0.0):
        super().__init__()
        def blk(i, o):
            layers = [nn.Conv2d(i, o, 3, padding=1, bias=not use_bn)]
            if use_bn:
                layers.append(nn.BatchNorm2d(o))   # argument is the CHANNEL count
            layers.append(nn.ReLU())               # order: conv -> BN -> ReLU
            return layers
        self.features = nn.Sequential(
            *blk(3, 32),  *blk(32, 32), nn.MaxPool2d(2),
            *blk(32, 64), *blk(64, 64), nn.MaxPool2d(2))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p_drop),                    # p = probability of ZEROING (PyTorch convention)
            nn.Linear(64 * 8 * 8, 512), nn.ReLU(),
            nn.Dropout(p_drop),                    # after the activation, never after the final layer
            nn.Linear(512, 10))
    def forward(self, x):
        return self.classifier(self.features(x))

def n_params(m):
    return sum(p.numel() for p in m.parameters())

for tag, m in [("plain", CNN()), ("dropout 0.5", CNN(p_drop=0.5)), ("BatchNorm", CNN(use_bn=True))]:
    print(f"{tag:12s} parameters = {n_params(m):,}")

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    """Mean cross-entropy and accuracy over a loader. Mirrors evaluate() in the scripts."""
    model.eval()                                   # <- switches dropout OFF and BatchNorm to running stats
    crit = nn.CrossEntropyLoss(reduction="sum")
    total = correct = 0
    loss_sum = 0.0
    for x, y in loader:
        x, y = x.to(dev), y.to(dev)
        out = model(x)
        loss_sum += crit(out, y).item()
        correct  += (out.argmax(1) == y).sum().item()
        total    += y.numel()
    return loss_sum / total, correct / total

def fit(model, epochs, lr=1e-3, log=True):
    """Adam at 1e-3, exactly as in the course experiments. Records train/test curves each epoch."""
    model = model.to(dev)
    opt = optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    hist = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}
    t0 = time.time()
    for ep in range(1, epochs + 1):
        model.train()                              # <- dropout / BatchNorm in training mode
        for x, y in train_loader:
            x, y = x.to(dev), y.to(dev)
            opt.zero_grad(); crit(model(x), y).backward(); opt.step()
        trl, tra = evaluate(model, train_loader)
        tel, tea = evaluate(model, test_loader)
        for k, v in zip(hist, (trl, tra, tel, tea)):
            hist[k].append(v)
        if log:
            print(f"epoch {ep:2d}  train_acc {tra:.3f}  test_acc {tea:.3f}  "
                  f"train_loss {trl:.3f}  test_loss {tel:.3f}")
    if log:
        print(f"done in {(time.time()-t0)/60:.1f} min")
    return hist

def plot_curves(hist, title):
    ep = range(1, len(hist["train_acc"]) + 1)
    fig, (a, b) = plt.subplots(1, 2, figsize=(11, 4))
    a.plot(ep, hist["train_acc"], "--", label="train")
    a.plot(ep, hist["test_acc"], "-", label="test")
    a.fill_between(ep, hist["train_acc"], hist["test_acc"], color="pink", alpha=.4)
    a.set_xlabel("epoch"); a.set_ylabel("accuracy"); a.set_title(title + " — accuracy")
    a.legend(); a.grid(alpha=.3)
    b.plot(ep, hist["train_loss"], "--", label="train loss")
    b.plot(ep, hist["test_loss"], "-", label="test loss")
    b.set_xlabel("epoch"); b.set_ylabel("cross-entropy"); b.set_title(title + " — loss")
    b.legend(); b.grid(alpha=.3)
    plt.tight_layout(); plt.show()

## 1. Overfitting

**Overfitting** happens when a model becomes so tuned to the training data that it captures not only the real pattern but also the noise and accidents of that particular sample. Those details are irrelevant, or actively misleading, on new data.

Train the plain Part 2 CNN and watch both curves. The reported run reaches **98.8% train against 74.3% test**, a **24.6-point generalization gap**. The right panel is the one to watch: **training loss falls smoothly toward zero, test loss falls for about four epochs and then turns around and climbs** (to 2.18 by epoch 30, roughly triple its best value). The loss curve turns *before* the accuracy curve, because loss registers growing over-confidence before any prediction actually flips — which is why **validation loss is the better early warning**.

`EPOCHS` defaults to 15 here to keep it quick; set it to 30 to reproduce the reported numbers.

In [ ]:
EPOCHS = 15   # the course numbers (98.8% / 74.3%) were measured at 30

plain = CNN()                        # no BatchNorm, no dropout
hist_plain = fit(plain, EPOCHS)
plot_curves(hist_plain, "Plain CNN")

gap = 100 * (hist_plain["train_acc"][-1] - hist_plain["test_acc"][-1])
print(f"\nfinal train {100*hist_plain['train_acc'][-1]:.1f}%   "
      f"test {100*hist_plain['test_acc'][-1]:.1f}%   gap {gap:.1f} pts")
print(f"best test loss at epoch {int(np.argmin(hist_plain['test_loss']))+1}, "
      f"final test loss {hist_plain['test_loss'][-1]:.2f}   "
      f"(reported run: gap 24.6 pts, test loss climbs to 2.18)")

## 2. Dropout

**Dropout randomly switches off some of the units in a hidden layer.** On every mini-batch a fresh random subset is disabled, so the network never trains the same architecture twice. Two reasons it helps: it **prevents co-adaptation** (a unit cannot rely on any other always being present), and it **trains an ensemble** of the `2**n` "thinned" sub-networks that share weights, which the full network approximates at test time.

*Papers:* Hinton et al. 2012 (arXiv:1207.0580, the one AlexNet cites) and Srivastava et al., "Dropout: A Simple Way to Prevent Neural Networks from Overfitting", JMLR 15 (2014), 1929–1958.

**The notation trap.** In the JMLR paper `p` is the probability a unit is *retained*; in PyTorch `nn.Dropout(p)` is the probability an element is *zeroed*. `nn.Dropout(0.5)` matches the paper's hidden-unit setting only because 0.5 is its own complement. PyTorch uses **inverted dropout**: it scales surviving activations by `1/(1-p)` during training so that `eval()` is a plain identity — which is why **`model.eval()` before validation is mandatory** (our `evaluate` already calls it).

Same architecture, same optimizer, same seed — only the two `nn.Dropout` layers go from `p=0` to `p=0.5`. Dropout adds **no parameters**. Reported result: **74.3% → 81.9%, a 7.6-point gain**, and its test loss never turns upward within 30 epochs (the dropout model has not started overfitting yet). Because it is deliberately handicapped each batch, it trains more slowly and usually wants more epochs.

In [ ]:
drop = CNN(p_drop=0.5)               # identical net, dropout 0.5 in the head
hist_drop = fit(drop, EPOCHS)
plot_curves(hist_drop, "CNN + dropout 0.5")

print(f"\nplain   : params {n_params(plain):,}   "
      f"test {100*hist_plain['test_acc'][-1]:.1f}%")
print(f"dropout : params {n_params(drop):,}   "
      f"test {100*hist_drop['test_acc'][-1]:.1f}%   (same parameter count)")
print("reported at 30 epochs:  plain 74.3%  ->  dropout 0.5  81.9%   (+7.6 pts, zero extra params)")

## 3. Batch Normalization

**Batch normalization normalizes each layer's output using the mean and variance of the current mini-batch**, so whatever the incoming distribution, what comes out has mean 0 and standard deviation 1. It then rescales by **learned** per-channel parameters `gamma` and `beta`, so the network keeps the final say and can even undo the normalization if that is best.

*Paper:* Ioffe & Szegedy, "Batch Normalization…", arXiv:1502.03167, ICML 2015. Its stated mechanism ("internal covariate shift") was later refuted by Santurkar et al. (NeurIPS 2018), who attribute the benefit to a **smoother optimization landscape**. A technique can be genuinely useful while the explanation attached to it is wrong.

`nn.BatchNorm2d(C)` takes the **channel count**: it computes `C` means and variances, each pooled over `N × H × W`, and holds `2C` learned parameters. Order is **conv → BN → ReLU**, with `bias=False` on that conv (BN cancels it). `model.eval()` matters here too: at test time BN uses a **running estimate** of the statistics rather than the batch.

Reported result: **74.3% → 78.0% for 384 extra parameters** (= `2 × (32+32+64+64)`). Note the right panel does *not* reproduce the paper's famous speed-up — our net is only four conv layers deep, so optimization was never the bottleneck. What we get is the side effect: batch-to-batch noise acting as a mild regularizer, so **slightly worse training loss but better test accuracy**. In the appendix, where nets are deep enough for optimization to be the real problem, BatchNorm is indispensable.

In [ ]:
bn = CNN(use_bn=True)                # BatchNorm after every conv, no dropout
hist_bn = fit(bn, EPOCHS)
plot_curves(hist_bn, "CNN + BatchNorm")

print(f"\nplain     : params {n_params(plain):,}   "
      f"train {100*hist_plain['train_acc'][-1]:.1f}%   test {100*hist_plain['test_acc'][-1]:.1f}%")
print(f"BatchNorm : params {n_params(bn):,}   "
      f"train {100*hist_bn['train_acc'][-1]:.1f}%   test {100*hist_bn['test_acc'][-1]:.1f}%")
print(f"extra parameters = {n_params(bn) - n_params(plain)}   (reported: 384; 74.3% -> 78.0%)")

## 4. Early Stopping

The cheapest regularizer of all requires no change to the model. Validation loss falls, reaches a minimum, and rises; **everything after that minimum is training that made the model worse.** Early stopping keeps a validation set, watches its loss, and stops when it stops improving.

In the reported run, validation loss bottoms out at **epoch 4** and test accuracy peaks at **76.2% at epoch 6**, while training the full 30 epochs delivered only **74.3%** — so the last 24 epochs cost accuracy *and* about five times the compute. Loss is the usual criterion because it turns earlier and more cleanly than accuracy.

Two rules that matter as much as the stopping itself:

1. **Roll back to the best state.** Keep the best `state_dict` and reload it; the last epoch is by definition one of the worse ones.
2. **Watch a *validation* split carved out of training data, never the test set.** Choosing the stopping epoch by test performance uses the test set to make a modelling decision, and its number is no longer an honest estimate.

Below we carve a 5,000-image validation split off the training set and use a **patience** parameter: keep going, remember the best model, and give up only after validation loss has failed to improve for `patience` epochs.

In [ ]:
# Carve a validation split off the TRAINING data (never touch the test set for stopping).
val_size = 5000
tr_idx, val_idx = torch.randperm(len(tr), generator=torch.Generator().manual_seed(0)).split(
    [len(tr) - val_size, val_size])
train_sub_loader = DataLoader(torch.utils.data.Subset(tr, tr_idx.tolist()),
                              batch_size=128, shuffle=True,  num_workers=2)
val_loader       = DataLoader(torch.utils.data.Subset(tr, val_idx.tolist()),
                              batch_size=512, shuffle=False, num_workers=2)

def fit_early_stop(model, max_epochs=40, patience=5, lr=1e-3):
    model = model.to(dev)
    opt = optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    best_loss, best_state, wait, best_epoch = float("inf"), None, 0, 0
    for ep in range(1, max_epochs + 1):
        model.train()
        for x, y in train_sub_loader:
            x, y = x.to(dev), y.to(dev)
            opt.zero_grad(); crit(model(x), y).backward(); opt.step()
        val_loss, val_acc = evaluate(model, val_loader)
        _, test_acc = evaluate(model, test_loader)
        marker = ""
        if val_loss < best_loss:
            best_loss, best_epoch, wait = val_loss, ep, 0
            best_state = copy.deepcopy(model.state_dict())   # remember the best
            marker = "  <- best"
        else:
            wait += 1
        print(f"epoch {ep:2d}  val_loss {val_loss:.3f}  val_acc {val_acc:.3f}  "
              f"test_acc {test_acc:.3f}{marker}")
        if wait >= patience:
            print(f"early stop at epoch {ep} (no val improvement for {patience} epochs)")
            break
    model.load_state_dict(best_state)                        # roll back to the best epoch
    _, test_acc = evaluate(model, test_loader)
    print(f"restored best epoch {best_epoch}; test acc there = {100*test_acc:.1f}%")
    return model

_ = fit_early_stop(CNN(), max_epochs=40, patience=5)

---

## Appendix: ResNet and the Residual Connection

A detour from regularization, here because the **residual connection** returns in Week 9 on Transformers. Meeting it now, in a setting you already understand, makes that topic easier.

### The degradation problem

A deeper network can always represent whatever a shallower one can — give the extra layers the identity mapping and you reproduce the shallow network exactly — so its best possible solution is at least as good. In practice it does not work out that way.

He et al., "Deep Residual Learning for Image Recognition" (arXiv:1512.03385, CVPR 2016), report that a plain 56-layer network has **higher *training* error** than a plain 20-layer one. The word "training" is doing the work: if it were overfitting it would show *lower* training error. This is an **optimization** failure, not a generalization one — adding layers made the network harder to train even though a good solution provably exists in its parameter space. The paper calls this **degradation**.

Reproduced on the course hardware at 30 epochs, no data augmentation, `nn.BatchNorm2d` after every convolution in every net:

| model | conv layers | parameters | **train** | test |
|---|---|---|---|---|
| plain | 8 | 187,082 | 96.4% | 81.3% |
| plain | 20 | 520,778 | 97.1% | 83.3% |
| **plain** | **56** | **1,521,866** | **87.1%** | **77.8%** |
| residual | 20 | 520,778 | 98.3% | 84.9% |
| **residual** | **56** | **1,521,866** | **98.1%** | **84.0%** |

Read the **train** column. Going 8 → 20 behaved as hoped. **Going to 56 layers broke it**: plain-56 training accuracy fell to 87.1%, a full 10 points below the 20-layer net, using 2.9× more parameters. Adding skip connections and changing nothing else took the same 56 layers to **98.1% training** at an **identical parameter count** — the entire difference is `+ x`.

One honest note: residual-56 (84.0% test) does not beat residual-20 (84.9%). At this scale 20 layers is already enough; what the residual connection did was not make depth *helpful* but stop it from being *harmful* — the precondition for the 152-layer nets that came later.

### The blocks — one addition apart

The `PlainBlock` / `ResBlock` / `make_net` below are copied verbatim from `experiments3.py` and `experiments4.py`. A plain block computes `F(x)`; a residual block adds the input back, `y = F(x) + x`, so the layers only have to learn the **change** to `x`. If the best move is to do nothing, driving `F` toward zero is easy, whereas making a stack of convs reproduce their own input is awkward. The addition also gives gradients a clean path backwards.

**`ResBlock` subclasses `PlainBlock` and overrides `forward` with a single `+ x`.** At each depth the plain and residual nets are literally the same class with the same parameter count — an identity shortcut is an addition and additions have no weights. (Degradation is not simply vanishing gradients, and BatchNorm does not fix it: every net here already has BatchNorm and the plain-56 degrades anyway.)

In [ ]:
class PlainBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.f = nn.Sequential(
            nn.Conv2d(c, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(),
            nn.Conv2d(c, c, 3, padding=1, bias=False), nn.BatchNorm2d(c))
        self.act = nn.ReLU()
    def forward(self, x):
        return self.act(self.f(x))                 # a plain block: no skip

class ResBlock(PlainBlock):
    """Identical to PlainBlock apart from the one addition in forward()."""
    def forward(self, x):
        return self.act(self.f(x) + x)             # the whole idea: + x

def make_net(n_blocks, block):
    """A stem, then n_blocks plain-or-residual blocks per stage, then a head."""
    layers = [nn.Conv2d(3, 32, 3, padding=1, bias=False), nn.BatchNorm2d(32), nn.ReLU()]
    for c_in, c_out in [(32, 32), (32, 64), (64, 64)]:
        if c_in != c_out:
            layers += [nn.Conv2d(c_in, c_out, 3, padding=1, bias=False),
                       nn.BatchNorm2d(c_out), nn.ReLU()]
        layers += [block(c_out) for _ in range(n_blocks)]
        layers += [nn.MaxPool2d(2)]
    layers += [nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, 10)]
    return nn.Sequential(*layers)

def conv_layers(m):
    return sum(1 for mod in m.modules() if isinstance(mod, nn.Conv2d))

for tag, net in [("plain-8",  make_net(1, PlainBlock)),
                 ("plain-20", make_net(3, PlainBlock)),
                 ("res-20",   make_net(3, ResBlock)),
                 ("plain-56", make_net(9, PlainBlock)),
                 ("res-56",   make_net(9, ResBlock))]:
    print(f"{tag:9s} conv_layers={conv_layers(net):2d}  params={n_params(net):,}")

### Runnable comparison: plain-20 vs residual-20

The depth-20 pair is the affordable version of the experiment (`n_blocks=3`). Same class, same depth, **520,778 parameters each** — the only difference is `ResBlock.forward` adding `x`. Even at 20 layers the residual net trains to higher accuracy (reported 98.3% vs 97.1% train, 84.9% vs 83.3% test). Bump `EPOCHS_DEPTH` toward 30 to sharpen the gap.

In [ ]:
EPOCHS_DEPTH = 15   # reported figures are at 30 epochs

def fit_seq(model, epochs, lr=1e-3):
    """Same training loop as fit(), for the nn.Sequential depth nets."""
    torch.manual_seed(0)                            # the scripts reseed before each model
    model = model.to(dev)
    opt = optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    for ep in range(1, epochs + 1):
        model.train()
        for x, y in train_loader:
            x, y = x.to(dev), y.to(dev)
            opt.zero_grad(); crit(model(x), y).backward(); opt.step()
        trl, tra = evaluate(model, train_loader)
        tel, tea = evaluate(model, test_loader)
        print(f"epoch {ep:2d}  train_acc {tra:.3f}  test_acc {tea:.3f}")
    return tra, tea

print("=== plain-20 ===")
p20_tr, p20_te = fit_seq(make_net(3, PlainBlock), EPOCHS_DEPTH)
print("\n=== residual-20 ===")
r20_tr, r20_te = fit_seq(make_net(3, ResBlock), EPOCHS_DEPTH)

print(f"\nplain-20     train {100*p20_tr:.1f}%   test {100*p20_te:.1f}%")
print(f"residual-20  train {100*r20_tr:.1f}%   test {100*r20_te:.1f}%")
print("reported at 30 epochs:  plain-20  97.1% / 83.3%    residual-20  98.3% / 84.9%")

### The full degradation experiment: plain-56 vs residual-56 (expensive)

This is where degradation is dramatic — plain-56 **87.1% train** against residual-56 **98.1%** at an identical parameter count — but 56 layers over 30 epochs is genuinely slow, minutes-to-tens-of-minutes per model even on a GPU. The cell below is guarded by `RUN_56 = False`; flip it to `True` (and ideally raise `EPOCHS_DEPTH`) when you have the time. It reuses `fit_seq` exactly, only the block count changes to `n_blocks=9`.

In [ ]:
RUN_56 = False   # set True to reproduce the headline degradation numbers (slow!)

if RUN_56:
    print("=== plain-56 ===")
    p56_tr, p56_te = fit_seq(make_net(9, PlainBlock), EPOCHS_DEPTH)
    print("\n=== residual-56 ===")
    r56_tr, r56_te = fit_seq(make_net(9, ResBlock), EPOCHS_DEPTH)
    print(f"\nplain-56     train {100*p56_tr:.1f}%   test {100*p56_te:.1f}%")
    print(f"residual-56  train {100*r56_tr:.1f}%   test {100*r56_te:.1f}%")
    print("reported at 30 epochs:  plain-56  87.1% / 77.8%    residual-56  98.1% / 84.0%")
    print("Both nets have 1,521,866 parameters; the entire difference is '+ x'.")
else:
    print("RUN_56 is False. The reported 30-epoch result:")
    print("  plain-56     train 87.1%   test 77.8%   (1,521,866 params)")
    print("  residual-56  train 98.1%   test 84.0%   (1,521,866 params, identical)")
    print("Plain-56 trains 11 points worse than residual-56 despite the same parameters:")
    print("that is degradation, an OPTIMIZATION failure the skip connection removes.")

## Recap

- **Overfitting** is the train/test gap: 98.8% train vs 74.3% test, 24.6 points. Watch **validation loss** — it turns upward first.
- **Dropout** (Srivastava et al., JMLR 2014) switches off a random subset of units per mini-batch. The paper's `p` is *retention*; PyTorch's `nn.Dropout(p)` is the *drop* rate, with inverted scaling that makes `model.eval()` mandatory. Result: **74.3% → 81.9%, +7.6 points, zero extra parameters.**
- **Early stopping** costs nothing: best test 76.2% at epoch 6 vs 74.3% at epoch 30. Use patience, keep the best `state_dict`, and select on a **validation** split — never the test set.
- **BatchNorm** (Ioffe & Szegedy, ICML 2015) normalises per channel over the mini-batch, then rescales by learned gamma/beta. Result: **74.3% → 78.0% for 384 parameters.** Its stated explanation (internal covariate shift) was refuted; the real benefit is a smoother landscape.
- **ResNet** (He et al., CVPR 2016) fixed **degradation** — a deeper plain net with higher *training* error. `y = F(x) + x` lets layers learn only the change, at no parameter cost. Our reproduction: plain-56 **87.1%** train vs residual-56 **98.1%**, identical parameter counts.
- The residual connection returns in **Week 9**: a Transformer's "Add and Norm" is `LayerNorm(x + Sublayer(x))`, and its paper cites He et al. at exactly that line.